In [9]:
import uproot
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from tqdm import tqdm

In [10]:
file_path = "Liquid_scin/DAQ/25C25_Cf_R6094_820V/UNFILTERED/Data_CH0@DT5730SB_27616_25C25_Cf_R6094_820V.root"
file = uproot.open(file_path)
tree = file['Data;4']
df = tree.arrays(library="np")
df = pd.DataFrame(df, columns=df.keys())

In [11]:
def preproc_samples(df):
    
    timestamp_right = 70
    timestamp_left = 40
    threshold = 0.3

    #calculate baseline
    filter_df = df[df['Flags'] == 16384].copy()
    samples = np.array(filter_df['Samples'].to_list())
    print(f"Count events before filter: {samples.shape[0]}")
    baseline = np.median(samples[:, :35], axis=1, keepdims=True)
    samples = -(samples - baseline)

    #to norm
    norm_samples = samples / np.max(samples, axis=1, keepdims=True)

    #filtering
    mask_right = np.max(norm_samples[:, timestamp_right:], axis=1, keepdims=False) > threshold
    mask_left = np.max(norm_samples[:, :timestamp_left], axis=1, keepdims=False) > threshold
    mask = mask_left | mask_right

    norm_samples = norm_samples[~mask]

    
    print(f"Count events after filter: {norm_samples.shape[0]}")

    return norm_samples, np.max(samples[~mask], axis=1), samples[~mask]


In [12]:
norm_samples, amp, samples = preproc_samples(df)

Count events before filter: 509204
Count events after filter: 508796


In [13]:
dataframe = pl.DataFrame(
    data={
        "amplitude":amp,
        "norm_samples":list(norm_samples),
        "samples": list(samples)
        })

In [14]:
dataframe.write_parquet("aaa.parquet")